# PCEP Block 4 — Functions and Exceptions

**PCEP-30-02 official block** (prerequisite exam, not PCAP) — 8 exam items, 28% of the PCEP exam.

Source: [Python Institute PCEP-30-02 syllabus](https://pythoninstitute.org/pcep-exam-syllabus).

---


## 4.1 Defining functions, return, None, recursion, generators

`def name(params):` defines a function; it only runs when *called*. `return value` sends a result back to the caller and immediately exits the function; a function with no `return` (or a bare `return`) implicitly returns `None`. **Recursion** is a function calling itself with a smaller version of the problem, until it reaches a *base case* that stops the recursion. A **generator** is a function that uses `yield` instead of `return` — each call to `next()` resumes the function right where it left off, producing one value at a time instead of building a whole list in memory up front.


In [ ]:
def factorial(n):
    if n <= 1:            # base case -- stops the recursion
        return 1
    return n * factorial(n - 1)   # recursive case: smaller subproblem

print(factorial(5))   # 5*4*3*2*1

def no_return():
    print('side effect only')

result = no_return()
print(result)          # None -- no return statement

def count_up_to(n):        # a GENERATOR function: uses yield, not return
    i = 1
    while i <= n:
        yield i
        i += 1

gen = count_up_to(3)
print(next(gen))    # 1 -- resumes right after the yield each time
print(next(gen))    # 2
print(next(gen))    # 3
print(list(count_up_to(5)))   # or just consume it all at once


## 4.2 Parameters vs. arguments, positional/keyword/default, scope

A **parameter** is the name in the function definition; an **argument** is the actual value passed at the call site. Arguments can be passed **positionally** (matched by order) or by **keyword** (matched by name, `func(x=1, y=2)`), or mixed (positional first, then keyword). Parameters can have **default values**, making them optional to pass. A variable assigned *inside* a function is **local** to it by default (this is called scope, and accidentally reusing an outer name inside a function is called *shadowing*); to modify a variable from the enclosing global scope, you must declare `global name` first.


In [ ]:
def greet(name, greeting='Hello'):   # greeting has a default value -> optional
    return f'{greeting}, {name}!'

print(greet('Ada'))                      # positional, uses the default greeting
print(greet('Ada', 'Hi'))                # positional for both
print(greet(name='Ada', greeting='Hey')) # keyword arguments
print(greet(greeting='Yo', name='Ada'))  # keyword args can be in ANY order

counter = 0
def increment_local():
    counter = 100          # this creates a NEW local variable -- shadows the outer one
    return counter

def increment_global():
    global counter          # explicitly refers to the outer/global counter
    counter += 1

print(increment_local(), counter)   # local didn't touch the global
increment_global()
increment_global()
print(counter)                       # global WAS changed this time


## 4.3 The built-in exception hierarchy

Every exception inherits from `BaseException`. Its direct children include `Exception` (the ancestor of essentially all errors you should `except`), plus two you almost never catch on purpose: `SystemExit` (raised by `sys.exit()`) and `KeyboardInterrupt` (raised when the user presses Ctrl+C). Under `Exception`, key branches for PCEP: `ArithmeticError` → `ZeroDivisionError`; `LookupError` → `IndexError` (bad sequence index) and `KeyError` (missing dict key); plus `TypeError` (wrong type for an operation) and `ValueError` (right type, invalid value). Because `except` matches a class *or any subclass*, catching `LookupError` also catches both `IndexError` and `KeyError`.


In [ ]:
print(IndexError.__bases__)     # -> (<class 'LookupError'>,)
print(KeyError.__bases__)        # -> (<class 'LookupError'>,)
print(ZeroDivisionError.__bases__)   # -> (<class 'ArithmeticError'>,)
print(issubclass(IndexError, LookupError))    # True
print(issubclass(IndexError, KeyError))        # False -- siblings, not related to each other

# Catching the PARENT class catches both children
for demo in [lambda: [1, 2][5], lambda: {'a': 1}['b']]:
    try:
        demo()
    except LookupError as e:
        print('Caught via LookupError:', type(e).__name__, '-', e)


## 4.4 try-except basics: ordering and propagation

`try` runs risky code; a matching `except` runs if that exact error type (or a superclass of it) is raised. With multiple `except` clauses, Python checks them top to bottom and stops at the **first match** — so order from most specific to most general. If a function raises an exception and doesn't catch it itself, the exception **propagates** up to whatever called that function, and keeps propagating until something catches it or the program crashes.


In [ ]:
def parse_age(text):
    return int(text)     # can raise ValueError if text isn't a valid integer

def get_age_safe(text):
    try:
        return parse_age(text)     # the ValueError, if any, propagates up from parse_age()
    except ValueError:
        print('Not a valid number, defaulting to 0')
        return 0

print(get_age_safe('30'))
print(get_age_safe('thirty'))    # parse_age() raises, get_age_safe() catches it

# Ordering except clauses: specific before general
try:
    result = [1, 2, 3][10]
except IndexError:                 # specific: matches first
    print('Index out of range')
except LookupError:                 # general (IndexError's parent) -- never reached here
    print('Some lookup error')
